# Llama-3.1-8B FP16 WikiText-2 Baseline

Evaluate the base model `meta-llama/Llama-3.1-8B` on the WikiText-2 test split with FP16 weights and activations. No quantization or CPU/disk offload is used.

Formal evaluation settings: **non-overlapping 2048-token blocks; the incomplete final block is dropped**. This matches the protocol used by the LLaMA-2 baseline notebooks in this project.

Requirements: an NVIDIA CUDA GPU with enough VRAM and Hugging Face access to the gated Llama 3.1 repository. A GPU with at least 24 GB VRAM is recommended.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import json
import os
import platform
import time
from getpass import getpass
from pathlib import Path

import datasets
import torch
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
OUTPUT_PATH = Path("llama3.1-8b-baseline-s2048.json")

if not torch.cuda.is_available():
    raise RuntimeError("This baseline requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"Model: {MODEL_ID}")
print(f"Protocol: {EVALUATION_PROTOCOL}")
print(f"Output: {OUTPUT_PATH}")

## Load the original FP16 model

Accept the Llama 3.1 license on Hugging Face and add a Colab secret named `HF_TOKEN`. The environment variable and a secure prompt are also supported. `device_map=0` places the entire model on GPU 0 and prevents automatic CPU/disk offload.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        pass
if not token:
    token = getpass("HF_TOKEN: ")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map=0,
    attn_implementation="eager",
    token=token,
)
model.eval()
model.config.use_cache = False

parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
parameter_devices = {p.device.type for p in model.parameters()}
assert parameter_dtypes == {torch.float16}, parameter_dtypes
assert parameter_devices == {"cuda"}, parameter_devices
assert not getattr(model, "is_quantized", False)

print(f"Model dtype: {next(model.parameters()).dtype}")
print(f"Model device: {next(model.parameters()).device}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Maximum context: {model.config.max_position_embeddings:,}")

## Prepare WikiText-2

The text joining method is kept identical to the existing LLaMA-2 baselines. Token counts may differ because Llama 3.1 uses a different tokenizer.

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(range(0, usable_length, stride), total=total_blocks, desc="Evaluating"):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()

        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

In [ ]:
metrics = evaluate_perplexity(model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER)
result = {
    "model": MODEL_ID,
    "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
    "split": SPLIT,
    "dtype": "float16",
    "quantized": False,
    "attention_implementation": "eager",
    "context_length": CONTEXT_LENGTH,
    "stride": STRIDE,
    "evaluation_protocol": EVALUATION_PROTOCOL,
    "drop_remainder": DROP_REMAINDER,
    "model_max_position_embeddings": model.config.max_position_embeddings,
    "gpu": torch.cuda.get_device_name(0),
    "cuda": torch.version.cuda,
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    **metrics,
}

print(json.dumps(result, indent=2, ensure_ascii=False))
OUTPUT_PATH.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {OUTPUT_PATH.resolve()}")

In [ ]:
from google.colab import files
files.download(str(OUTPUT_PATH))